# OneVoice V2 — MT benchmark
Chạy baseline hoặc checkpoint EnViT5 VI→EN đã fine-tune trên các suite cố định. Mỗi job lưu report vào Drive và in lỗi trực tiếp theo từng job.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'model_cache/huggingface')
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
# EnViT5's SentencePiece tokenizer is incompatible with Transformers 5.x.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy', 'PyYAML', 'torch', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0', 'sacremoses'], check=True)
REPORT_ROOT = DRIVE_ROOT / 'reports/mt'

def run_streaming(command, label):
    print(f'\n[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'[{label}] exit code: {code}', flush=True)
    if code:
        raise RuntimeError(f'{label} failed; the complete subprocess log is printed above.')

print('Source:', REPO, '| Reports:', REPORT_ROOT)


In [ ]:
JOBS = [(direction, suite, mode) for direction in ('vi2en', 'en2vi') for suite in ('test', 'minimal', 'safety') for mode in ('raw', 'context')]
for direction, suite, mode in JOBS:
    report_dir = REPORT_ROOT / direction / suite / mode
    label = f'MT {direction}/{suite}/{mode}'
    command = [sys.executable, 'scripts/benchmark_mt_v2.py', '--direction', direction, '--suite', suite, '--report-dir', str(report_dir)]
    if mode == 'context':
        command.append('--with-context')
    run_streaming(command, label)


In [ ]:
import json
{f'{direction}/{suite}/{mode}': json.loads((REPORT_ROOT / direction / suite / mode / 'aggregate.json').read_text(encoding='utf-8')) for direction in ('vi2en','en2vi') for suite in ('test','minimal','safety') for mode in ('raw','context')}
